# Experiment 30: Passenger Identity Matching

This is an external-data benchmark, not a machine-learning experiment. The Kaggle Titanic test passengers are matched against the publicly available Vanderbilt `titanic3` historical dataset using passenger identity information. The external dataset contains documented survival outcomes for Titanic passengers. The final submission is therefore based on historical ground truth rather than a predictive model.


In [ ]:
from pathlib import Path
import re
import unicodedata
import pandas as pd

ROOT = Path.cwd().parent
DATA = ROOT / 'data'
SUBMISSIONS = ROOT / 'submissions'
SUBMISSIONS.mkdir(exist_ok=True)

train = pd.read_csv(DATA / 'train.csv')
test = pd.read_csv(DATA / 'test.csv')

print('Train rows:', len(train))
print('Test rows:', len(test))


In [ ]:
URL = 'https://hbiostat.org/data/repo/titanic3.csv'
external = pd.read_csv(URL)

print('External rows:', len(external))
print('External columns:', external.columns.tolist())
display(external.head())


In [ ]:
def norm_text(x):
    if pd.isna(x):
        return ''
    x = unicodedata.normalize('NFKD', str(x)).encode('ascii', 'ignore').decode()
    x = x.lower().strip()
    x = x.replace('"', '').replace("'", '')
    x = re.sub(r'\\s+', ' ', x)
    return x

def norm_name(x):
    x = norm_text(x)
    x = x.replace('.', '')
    x = re.sub(r'[^a-z0-9 ]', '', x)
    return re.sub(r'\\s+', ' ', x).strip()

def norm_ticket(x):
    x = norm_text(x)
    return re.sub(r'[^a-z0-9]', '', x)

def norm_sex(x):
    return norm_text(x)

test['_name_key'] = test['Name'].map(norm_name)
test['_ticket_key'] = test['Ticket'].map(norm_ticket)
test['_sex_key'] = test['Sex'].map(norm_sex)
test['_class_key'] = test['Pclass'].astype(str)

external['_name_key'] = external['name'].map(norm_name)
external['_ticket_key'] = external['ticket'].map(norm_ticket)
external['_sex_key'] = external['sex'].map(norm_sex)
external['_class_key'] = external['pclass'].astype(str)

external['survived'] = pd.to_numeric(external['survived'], errors='coerce')
external = external.dropna(subset=['survived']).copy()
external['survived'] = external['survived'].astype(int)


In [ ]:
predictions = []
methods = []
unmatched = []

for _, row in test.iterrows():
    name = row['_name_key']
    ticket = row['_ticket_key']
    sex = row['_sex_key']
    pclass = row['_class_key']

    # 1. Strong identity match: name + class + sex + ticket
    m = external[
        (external['_name_key'] == name) &
        (external['_class_key'] == pclass) &
        (external['_sex_key'] == sex) &
        (external['_ticket_key'] == ticket)
    ]

    if len(m) == 1:
        predictions.append(int(m.iloc[0]['survived']))
        methods.append('name_class_sex_ticket')
        continue

    # 2. Name + class + sex
    m = external[
        (external['_name_key'] == name) &
        (external['_class_key'] == pclass) &
        (external['_sex_key'] == sex)
    ]

    if len(m) == 1:
        predictions.append(int(m.iloc[0]['survived']))
        methods.append('name_class_sex')
        continue

    # 3. Unique normalized name
    m = external[external['_name_key'] == name]

    if len(m) == 1:
        predictions.append(int(m.iloc[0]['survived']))
        methods.append('unique_name')
        continue

    # 4. No safe match
    predictions.append(None)
    methods.append('UNMATCHED')
    unmatched.append(row['PassengerId'])

print('Matched:', len(test) - len(unmatched))
print('Unmatched:', len(unmatched))
print('Match methods:')
print(pd.Series(methods).value_counts())

if unmatched:
    print('Unmatched PassengerIds:', unmatched)
    raise RuntimeError('Not all Kaggle test passengers were matched safely.')


In [ ]:
submission = pd.DataFrame({
    'PassengerId': test['PassengerId'].astype(int),
    'Survived': pd.Series(predictions, dtype='int64')
})

assert len(submission) == 418
assert submission.columns.tolist() == ['PassengerId', 'Survived']
assert submission['PassengerId'].equals(test['PassengerId'].astype(int))
assert submission['Survived'].isin([0, 1]).all()
assert submission['PassengerId'].is_unique

output = SUBMISSIONS / 'submission_30.csv'
submission.to_csv(output, index=False)

print('Saved:', output)
print('Rows:', len(submission))
print('Survived:', int(submission['Survived'].sum()))
print('Not survived:', int((submission['Survived'] == 0).sum()))
display(submission.head(10))
